# Predicción de Gravedad de Siniestros Viales — CABA
## Pipeline completo: Ingesta → Limpieza → Features → Modelo → Visualización

**Datasets utilizados (todos de data.buenosaires.gob.ar):**
1. Siniestros viales:
*   Víctimas homicidios
*   Víctimas lesiones
2. Flujo vehicular por radares AUSA (sensores autopista)

3. Flujo vehicular — Anillo Digital (sensores ciudad)

4. Registro de precipitaciones

5. Registro de temperatura


## 1. Imports y configuración global

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Visualización
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')

print("Librerías cargadas correctamente")


Librerías cargadas correctamente


## 2. Ingesta de datos

* Usamos gdown para descargar la carpeta `data/` desde un drive personal público en sólo lectura con los CSVs ya descargados del portal del gobierno de la ciudad

* Usaremos datos de 2021-2025


In [16]:
import os

# Descarga toda la carpeta compartida de una vez a la sesión de Colab
!pip install gdown -q
!gdown --folder "https://drive.google.com/drive/folders/1udLTw6AOKr7WZL1XBuHW7A5iKZZ97orW" -O .

Retrieving folder contents
Processing file 1x7xnuHSvm-h45cXVKgdRKuK5tniSl0rM dataset_flujo_vehicular.csv
Processing file 1L-mqlisQkOCXnQtZ1YUCUcqflT7t0Rdm flujo-vehicular-por-radares-2021.csv
Processing file 167CiOHNlDIAKJoPWCKDez3IupLcZf7vs flujo-vehicular-por-radares-2022.csv
Processing file 1AcbxOW1970WMav3SoGZgLijsxHRGoNvw flujo-vehicular-por-radares-2023.csv
Processing file 1lNh6Njz2e_KuIcXoKndcHtwUDh-5kRkj flujo-vehicular-por-radares-2024.csv
Processing file 1uq-9sUkKs44bQUocZ5ecgMO3WljT-kt5 flujo-vehicular-por-radares-2025.csv
Processing file 117T45ldEMUCMGsKFbB_TWapqBQmF5njy historico_precipitaciones.csv
Processing file 1fg2hofgEJiR2ZiYPntCFUrZZ3at-SHHt historico_temperaturas.csv
Processing file 15TnXbWmNoLMNEw8wk1xvXR_B3B52llv_ siniestros_viales_hechos.csv
Processing file 1hH1QNZkVF16Wk18A3aC1r84WE4_5CREY siniestros_viales_victimas.csv
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Failed to retrieve file url:

	Cannot 

In [ ]:
!rm -rf data/datos

In [ ]:
import os

# Función auxiliar de carga segura
def cargar_csv(path, nombre, encoding='latin-1', sep=','):
    try:
        df = pd.read_csv(path, encoding=encoding, sep=sep, low_memory=False)
        print(f"✅ {nombre}: {df.shape[0]:,} filas × {df.shape[1]} columnas")
        return df
    except FileNotFoundError:
        print(f"⚠️  {nombre}: archivo no encontrado en '{path}'. Saltando...")
        return None

# Carga de cada dataset
df_hechos       = cargar_csv('data/siniestros_viales_hechos.csv',                  'Siniestros — Hechos')
df_victimas     = cargar_csv('data/siniestros_viales_victimas.csv',                'Siniestros — Víctimas')
df_flujo_anillo = cargar_csv('data/dataset_flujo_vehicular.csv',                   'Flujo Anillo Digital')
df_ausa_2021    = cargar_csv('data/flujo-vehicular-por-radares-2021.csv',          'Flujo AUSA 2021')
df_ausa_2022    = cargar_csv('data/flujo-vehicular-por-radares-2022.csv',          'Flujo AUSA 2022')
df_ausa_2023    = cargar_csv('data/flujo-vehicular-por-radares-2023.csv',          'Flujo AUSA 2023')
df_ausa_2024    = cargar_csv('data/flujo-vehicular-por-radares-2024.csv',          'Flujo AUSA 2024')
df_ausa_2025    = cargar_csv('data/flujo-vehicular-por-radares-2025.csv',          'Flujo AUSA 2025')
df_lluvia       = cargar_csv('data/historico_precipitaciones.csv',                 'Precipitaciones')
df_temp         = cargar_csv('data/historico_temperaturas.csv',                    'Temperatura')

# Unificar los CSV de AUSA en uno solo
import pandas as pd
df_flujo_ausa = pd.concat(
    [df for df in [df_ausa_2021, df_ausa_2022, df_ausa_2023, df_ausa_2024, df_ausa_2025] if df is not None],
    ignore_index=True
)
print(f"✅ Flujo AUSA unificado: {df_flujo_ausa.shape[0]:,} filas × {df_flujo_ausa.shape[1]} columnas")


## 3. Exploración inicial (EDA rápido)

In [ ]:
def explorar(df, nombre):
    if df is None:
        print(f"⚠️  {nombre} no disponible")
        return
    print(f"\n{'='*55}")
    print(f"  📊 {nombre}")
    print(f"{'='*55}")
    print(f"  Forma        : {df.shape}")
    print(f"  Duplicados   : {df.duplicated().sum()}")
    print(f"\n  Columnas y tipos:")
    print(df.dtypes.to_string())
    print(f"\n  Nulos por columna:")
    nulos = df.isnull().sum()
    print(nulos[nulos > 0].to_string() if nulos.sum() > 0 else "  → Sin nulos")
    print(f"\n  Primeras filas:")
    display(df.head(3))

explorar(df_homicidios,   'Homicidios viales')
explorar(df_lesiones,     'Lesiones viales')
explorar(df_flujo_ausa,   'Flujo AUSA')
explorar(df_flujo_anillo, 'Flujo Anillo Digital')
explorar(df_lluvia,       'Precipitaciones')
explorar(df_temp,         'Temperatura')


## 4. Limpieza — Siniestros viales

### 4.1 Unificación de homicidios y lesiones

In [ ]:
def limpiar_siniestros(df_hom, df_les):
    if df_hom is None and df_les is None:
        print("⚠️  Sin datos de siniestros disponibles")
        return None

    frames = []

    if df_hom is not None:
        df_hom = df_hom.copy()
        df_hom['gravedad'] = 'FATAL'
        frames.append(df_hom)

    if df_les is not None:
        df_les = df_les.copy()
        # Intentamos inferir gravedad desde columna existente
        if 'gravedad' not in df_les.columns:
            df_les['gravedad'] = 'LESION'
        frames.append(df_les)

    df = pd.concat(frames, ignore_index=True)
    print(f"✅ Siniestros unificados: {df.shape[0]:,} registros")
    return df

df_siniestros = limpiar_siniestros(df_homicidios, df_lesiones)


### 4.2 Normalización de columnas clave

In [ ]:
def normalizar_siniestros(df):
    if df is None:
        return None
    df = df.copy()

    # ── Estandarizar nombres de columna ──────────────────
    df.columns = (df.columns
                  .str.strip()
                  .str.lower()
                  .str.replace(' ', '_')
                  .str.replace('ó','o').str.replace('é','e')
                  .str.replace('á','a').str.replace('í','i')
                  .str.replace('ú','u').str.replace('ñ','n'))

    print("Columnas disponibles:", list(df.columns))

    # ── Fecha ─────────────────────────────────────────────
    for col_fecha in ['fecha', 'fecha_hecho', 'fecha_siniestro']:
        if col_fecha in df.columns:
            df['fecha'] = pd.to_datetime(df[col_fecha], errors='coerce', dayfirst=True)
            df['año']   = df['fecha'].dt.year
            df['mes']   = df['fecha'].dt.month
            df['dia_semana'] = df['fecha'].dt.dayofweek  # 0=lunes
            df['es_fin_semana'] = df['dia_semana'].isin([5, 6]).astype(int)
            print(f"✅ Fecha procesada desde '{col_fecha}'")
            break

    # ── Hora ──────────────────────────────────────────────
    for col_hora in ['hora', 'hora_hecho', 'franja_hora']:
        if col_hora in df.columns:
            df['hora_num'] = pd.to_numeric(
                df[col_hora].astype(str).str.extract(r'(\d+)')[0],
                errors='coerce'
            )
            # Variables cíclicas (seno/coseno) para que el modelo entienda que 23hs ~ 0hs
            df['hora_sin'] = np.sin(2 * np.pi * df['hora_num'] / 24)
            df['hora_cos'] = np.cos(2 * np.pi * df['hora_num'] / 24)
            print(f"✅ Hora procesada desde '{col_hora}'")
            break

    # ── Coordenadas ───────────────────────────────────────
    for col_lat in ['lat', 'latitud', 'y']:
        if col_lat in df.columns:
            df['lat'] = (df[col_lat].astype(str)
                         .str.replace(',', '.', regex=False))
            df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
            break

    for col_lon in ['long', 'lon', 'longitud', 'x']:
        if col_lon in df.columns:
            df['lon'] = (df[col_lon].astype(str)
                         .str.replace(',', '.', regex=False))
            df['lon'] = pd.to_numeric(df['lon'], errors='coerce')
            break

    # ── Valores "Sin datos" → NaN ─────────────────────────
    sd_values = ['SD', 'S/D', 'Sin datos', 'sin datos', 'ND', 'N/D', '-', '']
    df.replace(sd_values, np.nan, inplace=True)

    # ── Texto a mayúsculas normalizadas ───────────────────
    for col in ['tipo_de_calle', 'rol', 'victima', 'acusado', 'sexo', 'gravedad']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.upper().str.strip()

    print(f"\n✅ Siniestros normalizados: {df.shape}")
    print(f"   Nulos restantes: {df.isnull().sum().sum()}")
    return df

df_siniestros = normalizar_siniestros(df_siniestros)


## 5. Limpieza — Flujo vehicular

In [ ]:
def limpiar_flujo(df, nombre):
    if df is None:
        return None
    df = df.copy()

    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(' ', '_')
                  .str.replace('ó','o').str.replace('é','e'))

    # Fecha y hora
    for col in ['fecha', 'fecha_hora']:
        if col in df.columns:
            df['fecha'] = pd.to_datetime(df[col], errors='coerce', dayfirst=True)
            df['año']   = df['fecha'].dt.year
            df['mes']   = df['fecha'].dt.month
            break

    if 'hora' in df.columns:
        df['hora_num'] = pd.to_numeric(df['hora'], errors='coerce')

    # Cantidad / volumen vehicular
    for col in ['cantidad', 'total', 'volumen', 'q']:
        if col in df.columns:
            df['volumen_vehicular'] = pd.to_numeric(df[col], errors='coerce')
            break

    # Coordenadas
    for col_lat in ['lat', 'latitud']:
        if col_lat in df.columns:
            df['lat'] = pd.to_numeric(
                df[col_lat].astype(str).str.replace(',', '.'), errors='coerce')
            break
    for col_lon in ['long', 'lon', 'longitud']:
        if col_lon in df.columns:
            df['lon'] = pd.to_numeric(
                df[col_lon].astype(str).str.replace(',', '.'), errors='coerce')
            break

    df.replace(['SD', 'S/D', '-', ''], np.nan, inplace=True)
    print(f"✅ {nombre} limpio: {df.shape}")
    return df

df_flujo_ausa   = limpiar_flujo(df_flujo_ausa,   'Flujo AUSA')
df_flujo_anillo = limpiar_flujo(df_flujo_anillo, 'Flujo Anillo Digital')


## 6. Limpieza — Datos climáticos

In [ ]:
def limpiar_clima(df, nombre):
    if df is None:
        return None
    df = df.copy()
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(' ', '_')
                  .str.replace('ó','o').str.replace('é','e')
                  .str.replace('á','a').str.replace('í','i'))

    # Año y mes
    for col in ['ano', 'año', 'year']:
        if col in df.columns:
            df['año'] = pd.to_numeric(df[col], errors='coerce')
            break
    for col in ['mes', 'month']:
        if col in df.columns:
            df['mes'] = pd.to_numeric(df[col], errors='coerce')
            break

    # Convertir numéricas
    for col in df.columns:
        if col not in ['año', 'mes']:
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(',', '.'), errors='coerce')

    df.replace(['SD', 'S/D', '-', ''], np.nan, inplace=True)
    print(f"✅ {nombre} limpio: {df.shape}")
    return df

df_lluvia = limpiar_clima(df_lluvia, 'Precipitaciones')
df_temp   = limpiar_clima(df_temp,   'Temperatura')


## 7. Integración de datasets (merge por fecha)

In [ ]:
def merge_datasets(df_sin, df_lluvia, df_temp, df_flujo_ausa, df_flujo_anillo):
    if df_sin is None:
        print("⚠️  Sin dataset principal de siniestros. Abortando merge.")
        return None

    df = df_sin.copy()

    # ── Merge con precipitaciones (por año + mes) ─────────
    if df_lluvia is not None and 'año' in df_lluvia.columns and 'mes' in df_lluvia.columns:
        df = df.merge(df_lluvia, on=['año', 'mes'], how='left', suffixes=('', '_lluvia'))
        print("✅ Merge con precipitaciones OK")

    # ── Merge con temperatura (por año + mes) ─────────────
    if df_temp is not None and 'año' in df_temp.columns and 'mes' in df_temp.columns:
        df = df.merge(df_temp, on=['año', 'mes'], how='left', suffixes=('', '_temp'))
        print("✅ Merge con temperatura OK")

    # ── Flujo AUSA: agregar volumen promedio por mes ───────
    if df_flujo_ausa is not None and 'volumen_vehicular' in df_flujo_ausa.columns:
        cols_agg = [c for c in ['año', 'mes'] if c in df_flujo_ausa.columns]
        if cols_agg:
            flujo_mensual = (df_flujo_ausa
                             .groupby(cols_agg)['volumen_vehicular']
                             .mean()
                             .reset_index()
                             .rename(columns={'volumen_vehicular': 'flujo_ausa_promedio'}))
            df = df.merge(flujo_mensual, on=cols_agg, how='left')
            print("✅ Merge con flujo AUSA OK")

    # ── Flujo Anillo: agregar volumen promedio por mes ─────
    if df_flujo_anillo is not None and 'volumen_vehicular' in df_flujo_anillo.columns:
        cols_agg = [c for c in ['año', 'mes'] if c in df_flujo_anillo.columns]
        if cols_agg:
            flujo_anillo_mensual = (df_flujo_anillo
                                    .groupby(cols_agg)['volumen_vehicular']
                                    .mean()
                                    .reset_index()
                                    .rename(columns={'volumen_vehicular': 'flujo_anillo_promedio'}))
            df = df.merge(flujo_anillo_mensual, on=cols_agg, how='left')
            print("✅ Merge con flujo Anillo Digital OK")

    print(f"\n📊 Dataset integrado final: {df.shape}")
    return df

df_final = merge_datasets(df_siniestros, df_lluvia, df_temp, df_flujo_ausa, df_flujo_anillo)


In [ ]:
if df_final is not None:
    print("Columnas del dataset integrado:")
    for col in df_final.columns:
        print(f"  {col}: {df_final[col].dtype}")


## 8. Feature Engineering

In [ ]:
def feature_engineering(df):
    if df is None:
        return None
    df = df.copy()

    # ── Franja horaria ────────────────────────────────────
    if 'hora_num' in df.columns:
        def franja(h):
            if pd.isna(h): return 'DESCONOCIDA'
            h = int(h)
            if 0  <= h < 6:  return 'MADRUGADA'
            if 6  <= h < 12: return 'MAÑANA'
            if 12 <= h < 18: return 'TARDE'
            return 'NOCHE'
        df['franja_horaria'] = df['hora_num'].apply(franja)

    # ── Estación del año ──────────────────────────────────
    if 'mes' in df.columns:
        def estacion(m):
            if pd.isna(m): return 'DESCONOCIDA'
            m = int(m)
            if m in [12, 1, 2]: return 'VERANO'
            if m in [3, 4, 5]:  return 'OTOÑO'
            if m in [6, 7, 8]:  return 'INVIERNO'
            return 'PRIMAVERA'
        df['estacion'] = df['mes'].apply(estacion)

    # ── Indicador de lluvia ───────────────────────────────
    for col_mm in ['mm', 'precipitacion_mm', 'milimetros']:
        if col_mm in df.columns:
            df['hubo_lluvia'] = (df[col_mm] > 0).astype(int)
            break

    # ── Log del flujo vehicular (reduce skewness) ─────────
    for col_f in ['flujo_ausa_promedio', 'flujo_anillo_promedio']:
        if col_f in df.columns:
            df[f'{col_f}_log'] = np.log1p(df[col_f])

    print(f"✅ Feature engineering completado. Shape: {df.shape}")
    return df

df_final = feature_engineering(df_final)


## 9. Análisis exploratorio visual

In [ ]:
if df_final is not None:
    # ── Distribución de gravedad ──────────────────────────
    if 'gravedad' in df_final.columns:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        df_final['gravedad'].value_counts().plot(
            kind='bar', ax=axes[0], color=['#e74c3c', '#e67e22', '#3498db'], edgecolor='black')
        axes[0].set_title('Distribución por Gravedad del Siniestro')
        axes[0].set_xlabel('Gravedad')
        axes[0].set_ylabel('Cantidad')
        axes[0].tick_params(axis='x', rotation=0)

        df_final['gravedad'].value_counts().plot(
            kind='pie', ax=axes[1], autopct='%1.1f%%',
            colors=['#e74c3c', '#e67e22', '#3498db'])
        axes[1].set_title('Proporción por Gravedad')
        axes[1].set_ylabel('')

        plt.tight_layout()
        plt.savefig('outputs/gravedad_distribucion.png', dpi=150, bbox_inches='tight')
        plt.show()


In [ ]:
if df_final is not None:
    # ── Siniestros por franja horaria ─────────────────────
    if 'franja_horaria' in df_final.columns:
        orden = ['MADRUGADA', 'MAÑANA', 'TARDE', 'NOCHE', 'DESCONOCIDA']
        conteo = df_final['franja_horaria'].value_counts().reindex(orden).dropna()

        plt.figure()
        conteo.plot(kind='bar', color='#2c3e50', edgecolor='white')
        plt.title('Siniestros por Franja Horaria')
        plt.xlabel('Franja')
        plt.ylabel('Cantidad')
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.savefig('outputs/siniestros_franja_horaria.png', dpi=150, bbox_inches='tight')
        plt.show()


In [ ]:
if df_final is not None:
    # ── Siniestros por mes ────────────────────────────────
    if 'mes' in df_final.columns:
        meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
        conteo_mes = df_final.groupby('mes').size()

        plt.figure()
        conteo_mes.plot(kind='bar', color='#16a085', edgecolor='white')
        plt.title('Siniestros por Mes')
        plt.xlabel('Mes')
        plt.ylabel('Cantidad')
        plt.xticks(ticks=range(12), labels=meses, rotation=0)
        plt.tight_layout()
        plt.savefig('outputs/siniestros_mes.png', dpi=150, bbox_inches='tight')
        plt.show()


In [ ]:
if df_final is not None:
    # ── Heatmap: día de semana × franja horaria ───────────
    if 'dia_semana' in df_final.columns and 'franja_horaria' in df_final.columns:
        dias = ['Lun','Mar','Mié','Jue','Vie','Sáb','Dom']
        pivot = df_final.groupby(['dia_semana', 'franja_horaria']).size().unstack(fill_value=0)
        pivot.index = [dias[i] for i in pivot.index if i < 7]

        plt.figure(figsize=(10, 5))
        sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5)
        plt.title('Siniestros: Día de semana × Franja horaria')
        plt.tight_layout()
        plt.savefig('outputs/heatmap_dia_franja.png', dpi=150, bbox_inches='tight')
        plt.show()


## 10. Mapa de calor geoespacial

In [ ]:
import importlib.util

if df_final is not None and 'lat' in df_final.columns and 'lon' in df_final.columns:
    if importlib.util.find_spec('folium') is not None:
        import folium
        from folium.plugins import HeatMap

        coords = df_final[['lat', 'lon']].dropna()
        # Filtrar coordenadas dentro de CABA (bounding box aproximado)
        coords = coords[
            (coords['lat'].between(-34.75, -34.52)) &
            (coords['lon'].between(-58.55, -58.33))
        ]

        m = folium.Map(location=[-34.61, -58.44], zoom_start=12, tiles='CartoDB positron')
        HeatMap(coords.values.tolist(), radius=10, blur=15, max_zoom=13).add_to(m)

        m.save('outputs/mapa_calor_siniestros.html')
        print("✅ Mapa guardado en outputs/mapa_calor_siniestros.html")
        print(f"   Coordenadas válidas usadas: {len(coords):,}")
        m
    else:
        print("⚠️  Folium no instalado. Ejecutá: pip install folium")
else:
    print("⚠️  Sin columnas lat/lon disponibles para el mapa")


## 11. Modelo predictivo — Pipeline de clasificación

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_auc_score)

import os
os.makedirs('outputs', exist_ok=True)


In [ ]:
def preparar_features(df):
    """Selecciona y prepara features y target para el modelo."""
    if df is None:
        return None, None, None, None

    TARGET = 'gravedad'
    if TARGET not in df.columns:
        print(f"⚠️  Columna '{TARGET}' no encontrada. Columnas disponibles: {list(df.columns)}")
        return None, None, None, None

    # Features numéricas potenciales
    FEAT_NUM = [c for c in [
        'hora_num', 'hora_sin', 'hora_cos', 'dia_semana', 'mes',
        'es_fin_semana', 'flujo_ausa_promedio', 'flujo_anillo_promedio',
        'flujo_ausa_promedio_log', 'flujo_anillo_promedio_log',
        'hubo_lluvia'
    ] if c in df.columns]

    # Features categóricas potenciales
    FEAT_CAT = [c for c in [
        'franja_horaria', 'estacion', 'tipo_de_calle',
        'victima', 'acusado', 'sexo'
    ] if c in df.columns]

    print(f"Features numéricas ({len(FEAT_NUM)}): {FEAT_NUM}")
    print(f"Features categóricas ({len(FEAT_CAT)}): {FEAT_CAT}")

    df_model = df[FEAT_NUM + FEAT_CAT + [TARGET]].dropna(subset=[TARGET])

    # Codificar target
    le = LabelEncoder()
    y = le.fit_transform(df_model[TARGET].astype(str))
    X = df_model[FEAT_NUM + FEAT_CAT]

    print(f"\nClases del target: {dict(zip(le.classes_, range(len(le.classes_))))}")
    print(f"Distribución: {pd.Series(y).value_counts().to_dict()}")
    print(f"Shape X: {X.shape}")

    return X, y, le, FEAT_NUM, FEAT_CAT

resultado = preparar_features(df_final)

if resultado[0] is not None:
    X, y, le, FEAT_NUM, FEAT_CAT = resultado


In [ ]:
if 'X' in dir() and X is not None:

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)

    # ── Preprocessors por tipo de columna ─────────────────
    num_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    cat_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    # Solo incluir transformadores para columnas que existen
    transformers = []
    if FEAT_NUM:
        transformers.append(('num', num_transformer, FEAT_NUM))
    if FEAT_CAT:
        transformers.append(('cat', cat_transformer, FEAT_CAT))

    preprocessor = ColumnTransformer(transformers=transformers)

    # ── Pipeline completo ─────────────────────────────────
    pipeline_rf = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=100, max_depth=10,
            class_weight='balanced', random_state=42, n_jobs=-1))
    ])

    pipeline_lr = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(
            max_iter=1000, class_weight='balanced', random_state=42))
    ])

    print("✅ Pipelines definidos")
    print("   → pipeline_rf: Random Forest")
    print("   → pipeline_lr: Regresión Logística (baseline)")


In [ ]:
if 'pipeline_rf' in dir():
    print("🔄 Entrenando modelos...")

    # Baseline: Regresión Logística
    pipeline_lr.fit(X_train, y_train)
    y_pred_lr = pipeline_lr.predict(X_test)
    print("\n── REGRESIÓN LOGÍSTICA (Baseline) ───────────────")
    print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

    # Modelo principal: Random Forest
    pipeline_rf.fit(X_train, y_train)
    y_pred_rf = pipeline_rf.predict(X_test)
    print("\n── RANDOM FOREST ────────────────────────────────")
    print(classification_report(y_test, y_pred_rf, target_names=le.classes_))


## 12. Evaluación del modelo

In [ ]:
if 'y_pred_rf' in dir():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Matriz de confusión — Random Forest
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, y_pred_rf),
        display_labels=le.classes_
    ).plot(ax=axes[0], colorbar=False, cmap='Blues')
    axes[0].set_title('Matriz de Confusión — Random Forest')

    # Matriz de confusión — Logística
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, y_pred_lr),
        display_labels=le.classes_
    ).plot(ax=axes[1], colorbar=False, cmap='Greens')
    axes[1].set_title('Matriz de Confusión — Logística (Baseline)')

    plt.tight_layout()
    plt.savefig('outputs/matrices_confusion.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
if 'pipeline_rf' in dir():
    # Feature importance del Random Forest
    rf_model = pipeline_rf.named_steps['classifier']
    prep = pipeline_rf.named_steps['preprocessor']

    # Obtener nombres de features post-transformación
    feature_names = []
    for name, transformer, cols in prep.transformers_:
        if name == 'num':
            feature_names.extend(cols)
        elif name == 'cat':
            ohe = transformer.named_steps['ohe']
            feature_names.extend(ohe.get_feature_names_out(cols))

    importances = pd.Series(rf_model.feature_importances_, index=feature_names)
    top15 = importances.nlargest(15)

    plt.figure(figsize=(10, 6))
    top15.sort_values().plot(kind='barh', color='#2980b9', edgecolor='white')
    plt.title('Top 15 Features más Importantes — Random Forest')
    plt.xlabel('Importancia')
    plt.tight_layout()
    plt.savefig('outputs/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()


## 13. Guardar resultados

In [ ]:
import pickle

# Dataset final limpio e integrado
if df_final is not None:
    df_final.to_csv('outputs/dataset_final_limpio.csv', index=False)
    print("✅ Dataset guardado en outputs/dataset_final_limpio.csv")

# Modelos entrenados
if 'pipeline_rf' in dir():
    with open('outputs/modelo_random_forest.pkl', 'wb') as f:
        pickle.dump(pipeline_rf, f)
    with open('outputs/modelo_logistica.pkl', 'wb') as f:
        pickle.dump(pipeline_lr, f)
    with open('outputs/label_encoder.pkl', 'wb') as f:
        pickle.dump(le, f)
    print("✅ Modelos guardados en outputs/")

print("\n🎉 Pipeline completo ejecutado exitosamente")


## 14. Resumen del Pipeline

```
[1] INGESTA          → Carga de 5 CSVs oficiales (GCBA)
       ↓
[2] EXPLORACIÓN      → Shape, tipos, nulos, duplicados
       ↓
[3] LIMPIEZA         → Normalización de columnas, fechas, coordenadas,
                       valores SD → NaN, texto a mayúsculas
       ↓
[4] INTEGRACIÓN      → Merge por año + mes con clima y flujo vehicular
       ↓
[5] FEATURE ENG.     → Franja horaria, estación, hubo_lluvia,
                       variables cíclicas hora (sin/cos), log flujo
       ↓
[6] EDA VISUAL       → Distribuciones, heatmap, mapa folium
       ↓
[7] MODELO           → ColumnTransformer + Pipeline sklearn
                       Baseline: Regresión Logística
                       Principal: Random Forest Classifier
       ↓
[8] EVALUACIÓN       → Classification report, matrices de confusión,
                       feature importance
       ↓
[9] PERSISTENCIA     → CSV limpio + modelos .pkl
```

### 📁 Archivos generados en `outputs/`
| Archivo | Descripción |
|---|---|
| `dataset_final_limpio.csv` | Dataset integrado y limpio |
| `mapa_calor_siniestros.html` | Mapa interactivo de accidentes |
| `gravedad_distribucion.png` | Distribución del target |
| `siniestros_franja_horaria.png` | Siniestros por hora del día |
| `siniestros_mes.png` | Siniestros por mes |
| `heatmap_dia_franja.png` | Heatmap día × franja |
| `matrices_confusion.png` | Evaluación de modelos |
| `feature_importance.png` | Variables más predictivas |
| `modelo_random_forest.pkl` | Modelo RF entrenado |
| `modelo_logistica.pkl` | Modelo LR entrenado |
| `label_encoder.pkl` | Encoder del target |
